# Semantic summaries with exact local interactions

Purpose: replace the encoder mixer with content-dependent normalized distant interactions and exact local exponential weights. The JEPA scaffold, trainable parameter shapes and predictor remain the ViT scaffold.

Inputs: packed visible features, original grid indices, validity, fixed seeded orthogonal Gaussian feature buffers. Outputs: aligned mixed features and `build_semantic_jepa(model_config)`.

Expected runtime: definitions only, under a few seconds after imports. This notebook never starts training. Checkpoints belong to the experiment orchestrator at `checkpoints/<run_id>/`; the feature bank is persisted in every model state. Exact resume: execute tagged definitions, reconstruct from the same model configuration, then restore complete model, optimizer, sampler and RNG state before taking a step. Only `ahr-export` cells are imported.

Each head uses positive features `exp(omega @ (q/d**0.25) - ||q/d**0.25||²/2)/sqrt(s)`. Within each orthogonal block, directions are Haar orthogonal and independently scaled by chi-distributed Gaussian row lengths. Buffers are fixed, not redrawn during training. This is a finite approximation to exponential dot-product weights; no uniform error guarantee is claimed.

Local means the same neighboring-leaf convention as the earlier FMM by default: 2×2-patch leaves and Chebyshev radius one, at most 36 original patch positions. Global summaries plus sparse replacement of approximate local weights produce one normalized hybrid kernel. Scalar shifts are applied equally to exact and approximate weights, without numerical clipping or denominator epsilon. Nonfinite active inputs, catastrophic negative far mass, or invalid denominators fail explicitly. Fully local receivers bypass subtractive roundoff by setting the mathematically zero far contribution to zero. Tiny reference and gradient checks live in `tests/test_semantic_attention.ipynb`.


In [ ]:
from pathlib import Path
import json, os
PROJECT_ROOT = Path(os.environ.get("FMM_JEPA_ROOT", "/home/benjamin/workspace/ahr_go/fmm_jeppa")).expanduser().resolve()
_bootstrap_namespace = {}
for _cell in json.loads((PROJECT_ROOT / "fmm_jepa/bootstrap.ipynb").read_text())["cells"]:
    if _cell["cell_type"] == "code" and "ahr-export" in _cell.get("metadata", {}).get("tags", []):
        exec(compile("".join(_cell["source"]), "bootstrap.ipynb#ahr-export", "exec"), _bootstrap_namespace)
_bootstrap_namespace["install_notebook_package"](PROJECT_ROOT)


In [ ]:
"""Content-dependent normalized global summaries with exact sparse local weights."""
import copy
import math
import torch
from torch import nn
from fmm_jepa.models import JEPA


def _orthogonal_gaussian_features(heads, features, head_dim, seed):
    """Orthogonal directions within each block, independent chi_d row radii."""
    generator = torch.Generator(device="cpu").manual_seed(int(seed))
    banks = []
    for _ in range(heads):
        blocks = []
        while sum(block.shape[0] for block in blocks) < features:
            gaussian = torch.randn(head_dim, head_dim, generator=generator, dtype=torch.float64)
            orthogonal, triangular = torch.linalg.qr(gaussian)
            signs = torch.where(torch.diag(triangular) >= 0, 1.0, -1.0)
            directions = (orthogonal * signs.unsqueeze(0)).T
            radii = torch.randn(head_dim, head_dim, generator=generator, dtype=torch.float64).norm(dim=1)
            blocks.append(directions * radii.unsqueeze(1))
        banks.append(torch.cat(blocks, dim=0)[:features])
    return torch.stack(banks).float()


class SemanticLocalMixer(nn.Module):
    """Approximate softmax far weights plus exact exponential near weights.

    qkv/projection parameters exactly match AttentionMixer. No learned FMM
    translations, no feature projections on invisible grid positions, and no
    dense token-pair matrix. The integer grid lookup has shape B*(N_grid+1).
    Locality follows the old FMM leaf convention by default: a 3x3 cell
    neighborhood of 2x2-patch leaves, at most 36 sources per receiver.
    """
    def __init__(self, dim, heads=4, semantic_features=32, local_leaf_size=2,
                 local_radius=1, feature_seed=20260925):
        super().__init__()
        if dim < 1 or heads < 1 or dim % heads:
            raise ValueError("positive dimension divisible by positive heads required")
        if semantic_features < 1 or local_leaf_size < 1 or local_radius < 0:
            raise ValueError("positive feature/leaf counts and nonnegative radius required")
        self.dim, self.heads = int(dim), int(heads)
        self.semantic_features = int(semantic_features)
        self.local_leaf_size, self.local_radius = int(local_leaf_size), int(local_radius)
        self.feature_seed = int(feature_seed)
        self.qkv = nn.Linear(dim, 3 * dim)
        self.projection = nn.Linear(dim, dim)
        self.register_buffer("omega", _orthogonal_gaussian_features(
            heads, semantic_features, dim // heads, feature_seed))
        self.backend = "positive orthogonal Gaussian features + sparse exact local exponential corrections"

    def _packed_neighbors(self, indices, valid, grid):
        """B,V,K packed-source indices and validity; no pairwise V,V structure."""
        batch, count = indices.shape
        rows, cols = map(int, grid)
        size, radius = self.local_leaf_size, self.local_radius
        # Invalid packed positions write only the dedicated sentinel grid slot.
        lookup = torch.full((batch, rows * cols + 1), -1, dtype=torch.long, device=indices.device)
        addresses = torch.where(valid, indices, rows * cols)
        positions = torch.arange(count, device=indices.device).expand(batch, -1)
        lookup.scatter_(1, addresses, torch.where(valid, positions, -1))
        if not bool((lookup.ge(0).sum(-1) == valid.sum(-1)).all()):
            raise ValueError("valid original positions must be unique within each sample")
        offsets = torch.arange(-radius, radius + 1, device=indices.device)
        within = torch.arange(size, device=indices.device)
        dy, dx, py, px = torch.meshgrid(offsets, offsets, within, within, indexing="ij")
        row = indices.clamp(0, rows * cols - 1).div(cols, rounding_mode="floor")
        col = indices.clamp(0, rows * cols - 1).remainder(cols)
        neighbor_row = (row.div(size, rounding_mode="floor")[..., None] + dy.flatten()) * size + py.flatten()
        neighbor_col = (col.div(size, rounding_mode="floor")[..., None] + dx.flatten()) * size + px.flatten()
        in_grid = ((neighbor_row >= 0) & (neighbor_row < rows) &
                   (neighbor_col >= 0) & (neighbor_col < cols) & valid[..., None])
        original = torch.where(in_grid, neighbor_row * cols + neighbor_col, rows * cols)
        packed = lookup.gather(1, original.flatten(1)).reshape_as(original)
        return packed.clamp_min(0), in_grid & (packed >= 0)

    @staticmethod
    def _gather_sources(tensor, neighbors):
        # B,H,V,D -> B,H,V,K,D; only bounded local source tensors are gathered.
        batch = torch.arange(tensor.shape[0], device=tensor.device)[:, None, None]
        return tensor.permute(0, 2, 1, 3)[batch, neighbors].permute(0, 3, 1, 2, 4)

    def mix_heads(self, q, k, v, indices, valid, grid):
        """Mix preprojected B,H,V,d tensors; public for independent numerical checks."""
        batch, heads, count, head_dim = q.shape
        if count == 0:
            return v * 0
        dtype = torch.float64 if q.dtype == torch.float64 else torch.float32
        q, k, v = q.to(dtype), k.to(dtype), v.to(dtype)
        active = valid[:, None, :, None]
        q, k, v = (torch.where(active, tensor, torch.zeros_like(tensor)) for tensor in (q, k, v))
        if not bool(torch.isfinite(q).all() & torch.isfinite(k).all() & torch.isfinite(v).all()):
            raise FloatingPointError("nonfinite projected values at valid semantic-attention positions")
        neighbors, neighbor_valid = self._packed_neighbors(indices, valid, grid)
        neighbor_k = self._gather_sources(k, neighbors)
        neighbor_v = self._gather_sources(v, neighbors)
        local_scores = (q.unsqueeze(-2) * neighbor_k).sum(-1) / math.sqrt(head_dim)
        pair_valid = neighbor_valid[:, None]
        local_scores = local_scores.masked_fill(~pair_valid, -torch.inf)
        scale = head_dim ** -0.25
        omega = self.omega.to(dtype)
        qs, ks = q * scale, k * scale
        normalizer = 0.5 * math.log(self.semantic_features)
        q_log = torch.einsum("bhvd,hsd->bhvs", qs, omega) - 0.5 * qs.square().sum(-1, keepdim=True) - normalizer
        k_log = torch.einsum("bhvd,hsd->bhvs", ks, omega) - 0.5 * ks.square().sum(-1, keepdim=True) - normalizer
        # One key shift for each batch/head, one query shift for each receiver.
        # Both approximate and exact weights receive exp(-q_shift-k_shift).
        # No clipping or epsilon denominator alters the represented operator.
        key_shift = k_log.masked_fill(~active, -torch.inf).amax(dim=(-2, -1), keepdim=True)
        key_shift = torch.where(valid.any(-1)[:, None, None, None], key_shift, torch.zeros_like(key_shift)).detach()
        query_shift = torch.maximum(q_log.amax(-1, keepdim=True),
                                    local_scores.amax(-1, keepdim=True) - key_shift).detach()
        q_features = torch.exp(q_log - query_shift)
        k_features = torch.where(active, torch.exp(k_log - key_shift), torch.zeros_like(k_log))
        summary = torch.matmul(k_features.transpose(-2, -1), v)
        total_num = torch.matmul(q_features, summary)
        total_den = (q_features * k_features.sum(-2, keepdim=True)).sum(-1)
        local_features = self._gather_sources(k_features, neighbors)
        local_approx = (q_features.unsqueeze(-2) * local_features).sum(-1)
        local_approx = torch.where(pair_valid, local_approx, torch.zeros_like(local_approx))
        local_exact = torch.exp(local_scores - query_shift - key_shift)
        local_exact = torch.where(pair_valid, local_exact, torch.zeros_like(local_exact))
        far_num = total_num - (local_approx.unsqueeze(-1) * neighbor_v).sum(-2)
        far_den = total_den - local_approx.sum(-1)
        # If all visible sources are local, far is mathematically identically zero.
        all_local = neighbor_valid.sum(-1).eq(valid.sum(-1, keepdim=True))[:, None]
        far_num = torch.where(all_local[..., None], torch.zeros_like(far_num), far_num)
        far_den = torch.where(all_local, torch.zeros_like(far_den), far_den)
        tolerance = 64 * torch.finfo(dtype).eps * total_den.detach().abs()
        if bool(((far_den < -tolerance) & valid[:, None]).any()):
            raise FloatingPointError("catastrophic cancellation in semantic far normalization")
        numerator = far_num + (local_exact.unsqueeze(-1) * neighbor_v).sum(-2)
        denominator = far_den + local_exact.sum(-1)
        healthy = torch.isfinite(numerator).all(-1) & torch.isfinite(denominator) & (denominator > 0)
        if not bool((healthy | ~valid[:, None]).all()):
            raise FloatingPointError("nonfinite or nonpositive semantic denominator; no silent clamp is used")
        safe_denominator = torch.where(valid[:, None], denominator, torch.ones_like(denominator))
        output = numerator / safe_denominator.unsqueeze(-1)
        return torch.where(active, output, torch.zeros_like(output))

    def forward(self, x, indices=None, valid=None, grid=None):
        batch, count, dim = x.shape
        if grid is None:
            raise ValueError("semantic attention requires the original patch grid")
        rows, cols = map(int, grid)
        if rows < 1 or cols < 1 or dim != self.dim:
            raise ValueError("positive original grid and matching feature dimension required")
        if indices is None:
            if count != rows * cols:
                raise ValueError("packed visible tokens require explicit original indices")
            indices = torch.arange(count, device=x.device).expand(batch, -1)
        if valid is None:
            valid = torch.ones_like(indices, dtype=torch.bool)
        if indices.shape != (batch, count) or valid.shape != indices.shape:
            raise ValueError("indices/valid must match B,V")
        if indices.dtype != torch.long or valid.dtype != torch.bool:
            raise TypeError("indices must be long and valid must be bool")
        if bool((((indices < 0) | (indices >= rows * cols)) & valid).any()):
            raise ValueError("valid original index outside grid")
        if count == 0:
            return x * 0
        clean = torch.where(valid.unsqueeze(-1), x, torch.zeros_like(x))
        projected = self.qkv(clean).reshape(batch, count, 3, self.heads, dim // self.heads).permute(2, 0, 3, 1, 4)
        q, k, v = projected.unbind(0)
        mixed = self.mix_heads(q, k, v, indices, valid, grid)
        mixed = mixed.transpose(1, 2).reshape(batch, count, dim).to(x.dtype)
        return torch.where(valid.unsqueeze(-1), self.projection(mixed), torch.zeros_like(x))


class SemanticJEPA(JEPA):
    """Original JEPA scaffold with only encoder mixers changed."""
    def __init__(self, config):
        configuration = copy.deepcopy(config)
        super().__init__({**configuration, "kind": "vit"})
        # Preserve global RNG: replacing modules must not alter subsequent masks,
        # augmentation, predictor initialization, or comparisons with the ViT.
        with torch.random.fork_rng(devices=[]):
            for layer, block in enumerate(self.context.blocks):
                original = block.mixer
                replacement = SemanticLocalMixer(
                    self.context.dim, configuration.get("heads", 4),
                    semantic_features=configuration.get("semantic_features", 32),
                    local_leaf_size=configuration.get("local_leaf_size", 2),
                    local_radius=configuration.get("local_radius", 1),
                    feature_seed=configuration.get("feature_seed", 20260925) + layer)
                replacement.qkv.load_state_dict(original.qkv.state_dict())
                replacement.projection.load_state_dict(original.projection.state_dict())
                block.mixer = replacement
        self.context.kind = "semantic"
        self.target = copy.deepcopy(self.context).requires_grad_(False).eval()
        self.config = {**configuration, "kind": "semantic"}

    @torch.no_grad()
    def update_teacher(self, momentum):
        super().update_teacher(momentum)
        # Fixed random features are a definition of the kernel, not an EMA state.
        # Copy after the base buffer policy to prevent accumulated roundoff drift.
        for context, target in zip(self.context.blocks, self.target.blocks):
            target.mixer.omega.copy_(context.mixer.omega)


def build_semantic_jepa(config):
    """Construct from a model-config subdictionary, not an entire run config."""
    return SemanticJEPA(config)
